In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime


## Função Para Ler a Partição

In [0]:
def ler_ultima_particao_delta(spark, base_path):
  """
  Essa fução é para ler a ultima partição dos volumes delta baseada na coluna 'data_processamento'
  """
  try: 
      # Descobrir as partições direto no storage 
      particoes = dbutils.fs.ls(base_path)
      datas = [
              int(p.name.split('=')[1].replace('/', '')) 
              for p in particoes if "data_processamento=" in p.name
      ]
      
      if not datas:
          print(f"Nenhuma partição encontrada em {base_path}")
          return None
      else:
          ultima_particao = max(datas)
          print(f"[{base_path}] Ultima partição: {ultima_particao}")
          return spark.read.format("delta").load(f"{base_path}/data_processamento={ultima_particao}")
  except Exception as e:
    print(f"Erro ao ler caminho {base_path}: {e}")
    return None

## Valores - Indicador Desempenho

### 1.1 tratemento silver

#### 1.1.1 LEITURA DOS DADOS BRONZE

In [0]:
bronze_path_selic = "/Volumes/workspace/case_spark_cvm/bronze/data_selic/"
df_bronze_selic = ler_ultima_particao_delta(spark, bronze_path_selic)

bronze_path_cdi = "/Volumes/workspace/case_spark_cvm/bronze/data_cdi_diario/"
df_bronze_cdi = ler_ultima_particao_delta(spark, bronze_path_cdi)

bronze_path_ipca = "/Volumes/workspace/case_spark_cvm/bronze/data_ipca_mensal/"
df_bronze_ipca = ler_ultima_particao_delta(spark, bronze_path_ipca)

bronze_path_ibov = "/Volumes/workspace/case_spark_cvm/bronze/data_ibov/"
df_bronze_ibov = ler_ultima_particao_delta(spark, bronze_path_ibov)

#### 1.1.2 TRATAMENTO SELIC, CDI E IBOV

In [0]:
df_selic = df_bronze_selic\
    .withColumn(
    "data",
    f.date_format(f.to_date(f.col("data"), "dd/MM/yyy"), "yyyy-MM-dd")
    )\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("valor", f.col("valor").cast(t.DecimalType(10,2)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("valor", "valor_selic")


df_cdi = df_bronze_cdi\
    .withColumn(
    "data",
    f.date_format(f.to_date(f.col("data"), "dd/MM/yyy"), "yyyy-MM-dd")
    )\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("valor", f.col("valor").cast(t.DecimalType(10,6)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("valor", "valor_cdi")


df_ibov = df_bronze_ibov\
    .withColumn(
    "data",
        f.from_unixtime(f.col("timestamp")).cast("date")
    )\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("close", f.col("close").cast(t.DecimalType(20,6)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("close", "ibov_close")\
    .drop("timestamp")\
    .select("data", "ibov_close", "data_processamento")

#### 1.1.3 TRATAMENTO IPCA (MENSAL) E CÁLCULO DO ACUMULADO (12 MESES)

In [0]:
df_bronze_ipca = df_bronze_ipca\
    .withColumn("data", f.date_format(f.to_date(f.col("data"), "dd/MM/yyy"), "yyyy-MM-dd"))\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("valor", f.col("valor").cast(t.DecimalType(10,4)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("valor", "ipca_mensal")\
    .withColumnRenamed("data", "data_ipca")
    
# Para calcular o IPCA acumulado de 12 meses corretamente (juros compostos)
# Fator = 1 + (ipca_mensal / 100)
df_ipca = df_bronze_ipca.withColumn("fator", (f.col("ipca_mensal") / 100) + 1)

# Usamos uma Window para pegar os últimos 12 meses ordenados pela data
# Acumulado = (Produto dos fatores de 12 meses) - 1. No PySpark: EXP(SUM(LOG(fator)))
window_12m = Window.orderBy("data_ipca").rowsBetween(-11, Window.currentRow)

df_ipca = df_ipca \
    .withColumn("fator_acumulado", f.exp(f.sum(f.log("fator")).over(window_12m))) \
    .withColumn("ipca_anual", ((f.col("fator_acumulado") - 1) * 100).cast(t.DecimalType(10, 2)))

# Criamos uma chave Ano-Mês para facilitar o join com os dados diários
df_ipca = df_ipca \
    .withColumn("ano_mes", f.date_format("data_ipca", "yyyy-MM")) \
    .select("ano_mes", "ipca_mensal", "ipca_anual")

#### 1.1.4 CRIAÇÃO DE UM CALENDÁRIO ÚNICO E JOIN DOS INDICADORES

In [0]:
# Extraímos todas as datas únicas disponíveis entre Selic e CDI

df_datas = df_selic.select("data").union(df_cdi.select("data")).distinct()

# Criamos a chave Ano-Mês nas datas base
df_datas = df_datas.withColumn("ano_mes", f.date_format("data", "yyyy-MM"))

# Realizamos o Join: left com Selic, left com CDI, left com IPCA
df_indicadores = df_datas \
    .join(df_selic, "data", "left") \
    .join(df_cdi, "data", "left") \
    .join(df_ibov, "data", "left")\
    .join(df_ipca, "ano_mes", "left")




#### 1.1.5 CONTORNO DO PROBLEMA DE IPCA ATRASADO (FORWARD FILL)

In [0]:
# Para os dias cujos meses ainda não têm IPCA lançado (Ex: fev/mar de 2026 ficarão nulos no join),
# preenchemos com o último valor de IPCA conhecido usando a função last() com ignorenulls=True.
window_ffill = Window.orderBy("data").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_indicadores = df_indicadores \
    .withColumn("ipca_mensal", f.last("ipca_mensal", ignorenulls=True).over(window_ffill)) \
    .withColumn("ipca_anual", f.last("ipca_anual", ignorenulls=True).over(window_ffill))


# Limpamos a tabela para o formato final e adicionamos data_processamento
data_proc = int(datetime.now().strftime("%Y%m%d"))

df_silver_indicadores = df_indicadores \
    .select("data", "valor_selic", "valor_cdi", "ipca_mensal", "ipca_anual", "ibov_close") \
    .withColumn("data_processamento", f.lit(data_proc).cast(t.IntegerType()))

#### 1.1.6 RETIRANDO DADOS DUPLICADOS DE DATA

In [0]:
df_silver_indicadores = df_silver_indicadores.dropDuplicates(["data"])

#### 1.1.7 TRATAMENTO DO TIPO DE DADO

In [0]:
df_silver_indicadores = df_silver_indicadores \
    .withColumn('data', f.col('data').cast(t.DateType())) \
    .withColumn('valor_selic', f.col('valor_selic').cast(t.DecimalType(10, 4))) \
    .withColumn('valor_cdi', f.col('valor_cdi').cast(t.DecimalType(10, 6))) \
    .withColumn('ipca_mensal', f.col('ipca_mensal').cast(t.DecimalType(10, 4))) \
    .withColumn('ipca_anual', f.col('ipca_anual').cast(t.DecimalType(10, 4))) \
    .withColumn('ibov_close', f.col('ibov_close').cast(t.DecimalType(18, 2))) \
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))

### 1.2 Salvar na camada Silver

In [0]:

df_silver_indicadores.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_dados_indicadores_economicos")

In [0]:
display(df_silver_indicadores.orderBy(f.col("data").desc()))